# PyTorch框架下Kernel直调

本节介绍自定义 Ascend C 算子接入 PyTorch 后进行 Kernel 直调的基本方法，示例会使用同一个 Ascend C Kernel 分别完成 pybind11 和 torch.library 两种调用路径。

本节学习大纲如下：
- 环境准备
- 准备可复用的 Ascend C Kernel
- pybind11 调用方式
- torch.library 调用方式
- 课后实践

---


## 1. 环境准备

本节示例需要先完成 PyTorch、torch_npu 和 pybind11 的安装与配置。

- [安装PyTorch框架和torch_npu插件](https://www.hiascend.com/document/detail/zh/Pytorch/latest/configandinstg/instg/docs/zh/installation_guide/installation_description.md)
- 安装 pybind11 和 expecttest：

    ```bash
    pip3 install pybind11 expecttest
    ```

在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及代码能编译运行。


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ASC_ARCH = "dav-2201"
ROOT = Path("src/02.06/kernel_pytorch_call")
COMMON_DIR = ROOT / "common"
PYBIND_DIR = ROOT / "pybind"
TORCH_LIBRARY_DIR = ROOT / "torch_library"
PRACTICE_DIR = ROOT / "practice_torch_library"
ANSWER_DIR = ROOT / "answer" / "02.06"

candidates = []
if os.environ.get("ASCEND_SET_ENV"):
    candidates.append(Path(os.environ["ASCEND_SET_ENV"]))
if os.environ.get("ASCEND_TOOLKIT_HOME"):
    candidates.append(Path(os.environ["ASCEND_TOOLKIT_HOME"]) / "set_env.sh")
candidates.append(Path("/usr/local/Ascend/cann/set_env.sh"))

CANN_SET_ENV = next((path for path in candidates if path.is_file()), None)
if CANN_SET_ENV is None:
    raise FileNotFoundError("未找到 CANN set_env.sh，请先安装 CANN Toolkit 或设置 ASCEND_SET_ENV。")

# 加载 CANN 环境并将变量导入 jupyter 进程，后续所有 ! 命令都会继承这些环境变量，
# 因此无需再在编译/运行脚本里单独 source set_env.sh。
result = subprocess.run(
    ["bash", "-lc", f"source {CANN_SET_ENV} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

# 固定使用当前 jupyter 内核的 Python 解释器（torch/torch_npu/pybind11 都装在这里）。
# 后续 cmake 用 -DPython3_EXECUTABLE=$PY_BIN 指定它，测试也用 $PY_BIN 运行，
# 避免 cmake 的 find_package(Python3) 误选到 PATH 上没有 torch 的其它 conda 环境（如 base）。
os.environ["PY_BIN"] = sys.executable

if ROOT.exists():
    shutil.rmtree(ROOT)
for directory in [COMMON_DIR, PYBIND_DIR, TORCH_LIBRARY_DIR, PRACTICE_DIR, ANSWER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"CANN环境脚本: {CANN_SET_ENV}")
print(f"编译架构: {ASC_ARCH}")
print(f"Python解释器: {sys.executable}")
print(f"课程代码目录: {ROOT}")
print("Environment initialization process completed successfully.")

检查 PyTorch、torch_npu 和 pybind11 是否可以正常导入。CANN 环境变量已在上一步导入当前进程，后续 `!` 命令会直接继承，因此直接运行 Python 检查即可。


In [ ]:
!$PY_BIN -c "import torch, torch_npu, pybind11; \
print('torch:', torch.__version__); \
print('torch_npu:', torch_npu.__version__); \
print('pybind11:', pybind11.__version__); \
print('NPU可用数量:', torch.npu.device_count())"

### 准备可复用的 Ascend C Kernel

pybind11 和 torch.library 的差异主要在 PyTorch 封装层。本节使用同一个 `add_custom.asc`，并在其中提供 `extern "C" void add_custom_impl(...)` 作为 C++ 封装层调用的入口。Kernel 内部按照 TPipe/TQue 编程范式完成 GM 到 UB 的搬入、UB 内向量加法和结果写回 GM。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/common/add_custom.asc
#include "kernel_operator.h"

__global__ __vector__ void add_custom(__gm__ uint8_t* x, __gm__ uint8_t* y, __gm__ uint8_t* z,
                                      uint32_t totalLength)
{
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueueZ;
    AscendC::GlobalTensor<half> xGm;
    AscendC::GlobalTensor<half> yGm;
    AscendC::GlobalTensor<half> zGm;

    uint32_t blockLength = totalLength / AscendC::GetBlockNum();
    xGm.SetGlobalBuffer((__gm__ half*)x + blockLength * AscendC::GetBlockIdx(), blockLength);
    yGm.SetGlobalBuffer((__gm__ half*)y + blockLength * AscendC::GetBlockIdx(), blockLength);
    zGm.SetGlobalBuffer((__gm__ half*)z + blockLength * AscendC::GetBlockIdx(), blockLength);

    pipe.InitBuffer(inQueueX, 1, blockLength * sizeof(half));
    pipe.InitBuffer(inQueueY, 1, blockLength * sizeof(half));
    pipe.InitBuffer(outQueueZ, 1, blockLength * sizeof(half));

    AscendC::LocalTensor<half> xLocal = inQueueX.AllocTensor<half>();
    AscendC::LocalTensor<half> yLocal = inQueueY.AllocTensor<half>();
    AscendC::DataCopy(xLocal, xGm, blockLength);
    AscendC::DataCopy(yLocal, yGm, blockLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);

    xLocal = inQueueX.DeQue<half>();
    yLocal = inQueueY.DeQue<half>();
    AscendC::LocalTensor<half> zLocal = outQueueZ.AllocTensor<half>();
    AscendC::Add(zLocal, xLocal, yLocal, blockLength);
    outQueueZ.EnQue<half>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);

    zLocal = outQueueZ.DeQue<half>();
    AscendC::DataCopy(zGm, zLocal, blockLength);
    outQueueZ.FreeTensor(zLocal);
}

extern "C" void add_custom_impl(uint32_t numBlocks, void* aclStream, uint8_t* x, uint8_t* y, uint8_t* z,
                                uint32_t totalLength)
{
    add_custom<<<numBlocks, 0, aclStream>>>(x, y, z, totalLength);
}


## 2. 如何使用pybind调用算子

pybind11 方式的目标是把 C++ 函数封装成 Python 模块函数。Python 侧导入模块后，像调用普通 Python 函数一样调用 Ascend C Kernel 封装函数。

调用链路如下：

`Python import ascendc_ops` -> `ascendc_ops.ascendc_add(...)` -> C++ 封装函数 -> `add_custom_impl(...)` -> `add_custom<<<...>>>(...)`


### 2.1 pybind封装代码编写

pybind 封装代码由三部分组成：

1. 引入 pybind11、PyTorch 与 torch_npu 相关头文件；
2. 编写面向 PyTorch Tensor 的 C++ 调用函数；
3. 使用 `PYBIND11_MODULE` 将 C++ 函数绑定到 Python 模块。


#### 2.1.1 头文件引用

`torch/extension.h` 提供 `at::Tensor` 等 PyTorch C++ API；`NPUStream.h` 用于获取当前 NPU stream；`pybind11/pybind11.h` 用于声明 Python 扩展模块。`add_custom_impl` 是公共 Kernel 文件提供的 C 接口。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/pybind/pybind_binding.cpp
#include <cstdint>
#include <pybind11/pybind11.h>
#include <torch/extension.h>
#include "torch_npu/csrc/core/npu/NPUStream.h"

extern "C" void add_custom_impl(uint32_t numBlocks, void* aclStream, uint8_t* x, uint8_t* y, uint8_t* z,
                                uint32_t totalLength);


#### 2.1.2 算子调用函数编写

封装函数的输入输出使用 `at::Tensor`。函数内部完成三件事：

- 通过 `c10_npu::getCurrentNPUStream()` 取得当前 NPU stream；
- 使用 `at::empty_like(x)` 在 NPU 侧创建输出 Tensor；
- 取出 Tensor 的 device 指针，并调用 `add_custom_impl(...)` 发起 Kernel 执行。


In [ ]:
%%writefile -a src/02.06/kernel_pytorch_call/pybind/pybind_binding.cpp

namespace ascendc_ops {
at::Tensor ascendc_add(const at::Tensor& x, const at::Tensor& y)
{
    auto aclStream = c10_npu::getCurrentNPUStream().stream(false);
    at::Tensor z = at::empty_like(x);

    uint32_t numBlocks = 8;
    uint32_t totalLength = 1;
    for (uint32_t size : x.sizes()) {
        totalLength *= size;
    }

    add_custom_impl(numBlocks, aclStream, reinterpret_cast<uint8_t*>(x.mutable_data_ptr()),
                    reinterpret_cast<uint8_t*>(y.mutable_data_ptr()),
                    reinterpret_cast<uint8_t*>(z.mutable_data_ptr()), totalLength);
    return z;
}
}  // namespace ascendc_ops


#### 2.1.3 绑定算子调用函数

`PYBIND11_MODULE(ascendc_ops, m)` 定义 Python 模块名为 `ascendc_ops`。`m.def(...)` 将 C++ 函数 `ascendc_ops::ascendc_add` 暴露成 Python 函数 `ascendc_add`。


In [ ]:
%%writefile -a src/02.06/kernel_pytorch_call/pybind/pybind_binding.cpp

PYBIND11_MODULE(ascendc_ops, m)
{
    m.doc() = "Ascend C add_custom pybind11 interfaces";
    m.def("ascendc_add", &ascendc_ops::ascendc_add, "Call add_custom Ascend C kernel");
}


#### 2.1.4 编译算子so

pybind11 路径会编译出两个动态库：

- `libascendc_kernels.so`：由 `add_custom.asc` 编译得到，包含 Kernel 与 `add_custom_impl`；
- `ascendc_ops*.so`：Python 扩展模块，封装 `ascendc_add`。

CMake 中通过 Python 查询 Torch、torch_npu、pybind11 的安装路径，并用 `--npu-arch=dav-2201` 编译 `.asc` 文件。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/pybind/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture")
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_LIBRARY_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR})

find_package(ASC REQUIRED)
project(kernel_pytorch_call_pybind LANGUAGES ASC CXX)
find_package(Python3 COMPONENTS Interpreter Development REQUIRED)

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import torch; print(torch.utils.cmake_prefix_path)"
    OUTPUT_VARIABLE TORCH_CMAKE_PREFIX_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
    RESULT_VARIABLE TORCH_QUERY_RESULT
)
if(NOT TORCH_QUERY_RESULT EQUAL 0)
    message(FATAL_ERROR "Failed to query Torch CMake prefix")
endif()
string(REPLACE "\r" "\n" TORCH_CMAKE_PREFIX_PATH_LINES "${TORCH_CMAKE_PREFIX_PATH_RAW}")
string(REGEX MATCH "[^\n]+$" TORCH_CMAKE_PREFIX_PATH "${TORCH_CMAKE_PREFIX_PATH_LINES}")
find_package(Torch REQUIRED HINTS ${TORCH_CMAKE_PREFIX_PATH})

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import os, torch_npu; print(os.path.dirname(torch_npu.__file__))"
    OUTPUT_VARIABLE TORCH_NPU_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
    RESULT_VARIABLE TORCH_NPU_QUERY_RESULT
)
if(NOT TORCH_NPU_QUERY_RESULT EQUAL 0)
    message(FATAL_ERROR "Failed to query torch_npu path")
endif()
string(REPLACE "\r" "\n" TORCH_NPU_PATH_LINES "${TORCH_NPU_PATH_RAW}")
string(REGEX MATCH "[^\n]+$" TORCH_NPU_PATH "${TORCH_NPU_PATH_LINES}")
set(TORCH_NPU_INCLUDE_DIRS ${TORCH_NPU_PATH}/include)
set(TORCH_NPU_LIBRARIES ${TORCH_NPU_PATH}/lib)

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import pybind11; print(pybind11.get_cmake_dir())"
    OUTPUT_VARIABLE PYBIND11_CMAKE_PREFIX_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
    RESULT_VARIABLE PYBIND11_QUERY_RESULT
)
if(NOT PYBIND11_QUERY_RESULT EQUAL 0)
    message(FATAL_ERROR "Failed to query pybind11 CMake path")
endif()
string(REPLACE "\r" "\n" PYBIND11_CMAKE_PREFIX_PATH_LINES "${PYBIND11_CMAKE_PREFIX_PATH_RAW}")
string(REGEX MATCH "[^\n]+$" PYBIND11_CMAKE_PREFIX_PATH "${PYBIND11_CMAKE_PREFIX_PATH_LINES}")
find_package(pybind11 REQUIRED HINTS ${PYBIND11_CMAKE_PREFIX_PATH})

add_library(ascendc_kernels SHARED
    ${CMAKE_CURRENT_LIST_DIR}/../common/add_custom.asc
)

pybind11_add_module(ascendc_ops SHARED
    pybind_binding.cpp
)

target_include_directories(ascendc_ops PRIVATE
    ${TORCH_INCLUDE_DIRS}
    ${TORCH_NPU_INCLUDE_DIRS}
)

target_link_directories(ascendc_ops PRIVATE
    ${TORCH_NPU_LIBRARIES}
)

target_link_libraries(ascendc_ops PRIVATE
    ascendc_kernels
    torch_npu
)

target_compile_options(ascendc_kernels PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

target_compile_options(ascendc_ops PRIVATE
    ${TORCH_CXX_FLAGS}
)

set_target_properties(ascendc_ops PROPERTIES
    BUILD_RPATH "$ORIGIN"
    INSTALL_RPATH "$ORIGIN"
)


下面直接完成 pybind11 样例的配置、编译和运行。CANN 环境变量已导入当前 jupyter 进程，可直接调用 `cmake` 与 `make`；编译产物在 `build` 目录下，运行测试前把 `build` 目录加入 `LD_LIBRARY_PATH` 即可。


#### 2.1.5 python侧调用代码

pybind11 方式在 Python 侧的关键动作是：

1. 将当前运行目录加入 `sys.path`；
2. `import ascendc_ops` 导入扩展模块；
3. 将输入 Tensor 放到 NPU；
4. 调用 `ascendc_ops.ascendc_add(...)`；
5. 将结果搬回 CPU，与 `torch.add` 的结果比较。

本节会进入 `build` 目录执行测试，因此 Python 侧可以和参考样例一样通过 `sys.path.append(os.getcwd())` 找到编译生成的 `ascendc_ops` 扩展模块。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/pybind/add_custom_test.py
import os
import sys

import torch
import torch_npu
from torch_npu.testing.testcase import TestCase, run_tests

sys.path.append(os.getcwd())
import ascendc_ops


class TestPybindCustomAdd(TestCase):
    def test_add_custom_ops(self):
        torch.manual_seed(0)
        shape = (8, 2048)
        x = torch.rand(shape, device="cpu", dtype=torch.float16)
        y = torch.rand(shape, device="cpu", dtype=torch.float16)

        output = ascendc_ops.ascendc_add(x.npu(), y.npu()).cpu()
        expected = torch.add(x, y)
        self.assertRtolEqual(output, expected)


if __name__ == "__main__":
    run_tests()


In [ ]:
!cd src/02.06/kernel_pytorch_call/pybind && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 -DPython3_EXECUTABLE="$PY_BIN" .. && \
make -j && \
LD_LIBRARY_PATH="$PWD:${LD_LIBRARY_PATH:-}" "$PY_BIN" ../add_custom_test.py

## 3. 如何使用torch.library调用算子

torch.library 方式不是把函数直接导出为普通 Python 模块函数，而是把自定义算子注册到 PyTorch 的算子系统。Python 侧加载动态库后，可以通过 `torch.ops.<namespace>.<op_name>` 调用。

调用链路如下：

`torch.ops.load_library(...)` -> `torch.ops.ascendc_ops.ascendc_add(...)` -> PyTorch dispatcher -> NPU 实现函数 -> `add_custom_impl(...)` -> `add_custom<<<...>>>(...)`


### 3.1 torch.library封装代码编写

torch.library 封装代码同样先编写 `at::Tensor` 版本的 C++ 调用函数，再额外完成两个注册动作：

- 使用 `TORCH_LIBRARY` 定义 namespace 和 schema；
- 使用 `TORCH_LIBRARY_IMPL` 将 NPU 设备实现注册到 `PrivateUse1`。


#### 3.1.1 头文件引用

torch.library 路径不需要 `pybind11/pybind11.h`。它需要 PyTorch C++ API 与 torch_npu stream API。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/torch_library/torch_binding.cpp
#include <cstdint>
#include <torch/extension.h>
#include "torch_npu/csrc/core/npu/NPUStream.h"

extern "C" void add_custom_impl(uint32_t numBlocks, void* aclStream, uint8_t* x, uint8_t* y, uint8_t* z,
                                uint32_t totalLength);

namespace ascendc_ops {
at::Tensor ascendc_add(const at::Tensor& x, const at::Tensor& y)
{
    auto aclStream = c10_npu::getCurrentNPUStream().stream(false);
    at::Tensor z = at::empty(x.sizes(), x.options());

    uint32_t numBlocks = 8;
    uint32_t totalLength = 1;
    for (uint32_t size : x.sizes()) {
        totalLength *= size;
    }

    add_custom_impl(numBlocks, aclStream, reinterpret_cast<uint8_t*>(x.mutable_data_ptr()),
                    reinterpret_cast<uint8_t*>(y.mutable_data_ptr()),
                    reinterpret_cast<uint8_t*>(z.mutable_data_ptr()), totalLength);
    return z;
}
}  // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m)
{
    m.def("ascendc_add(Tensor x, Tensor y) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m)
{
    m.impl("ascendc_add", TORCH_FN(ascendc_ops::ascendc_add));
}


#### 3.1.2 算子调用函数编写

这部分和 pybind11 路径基本一致：输入输出仍然是 `at::Tensor`，仍然通过当前 NPU stream 发起 `add_custom_impl(...)` 调用。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/torch_library/torch_binding_excerpt_call.cpp
namespace ascendc_ops {
at::Tensor ascendc_add(const at::Tensor& x, const at::Tensor& y)
{
    auto aclStream = c10_npu::getCurrentNPUStream().stream(false);
    at::Tensor z = at::empty(x.sizes(), x.options());
    uint32_t numBlocks = 8;
    uint32_t totalLength = 1;
    for (uint32_t size : x.sizes()) {
        totalLength *= size;
    }
    add_custom_impl(numBlocks, aclStream, reinterpret_cast<uint8_t*>(x.mutable_data_ptr()),
                    reinterpret_cast<uint8_t*>(y.mutable_data_ptr()),
                    reinterpret_cast<uint8_t*>(z.mutable_data_ptr()), totalLength);
    return z;
}
}  // namespace ascendc_ops


#### 3.1.3 自定义算子的注册

`TORCH_LIBRARY(ascendc_ops, m)` 创建自定义算子 namespace，并声明 `ascendc_add(Tensor x, Tensor y) -> Tensor`。

`TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m)` 把 NPU 设备上的实现绑定到上面的 schema。torch_npu 将 NPU 设备接入 PyTorch 时使用 `PrivateUse1` 作为调度键。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/torch_library/torch_binding_excerpt_register.cpp
TORCH_LIBRARY(ascendc_ops, m)
{
    m.def("ascendc_add(Tensor x, Tensor y) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m)
{
    m.impl("ascendc_add", TORCH_FN(ascendc_ops::ascendc_add));
}


#### 3.1.4 编译算子so

torch.library 路径同样编译两个动态库：

- `libascendc_kernels.so`：公共 Ascend C Kernel 库；
- `libascendc_ops.so`：包含 torch.library 注册逻辑的动态库。

Python 侧加载 `libascendc_ops.so` 时，库中的 `TORCH_LIBRARY` 注册代码会执行，自定义算子随即可以通过 `torch.ops.ascendc_ops.ascendc_add` 调用。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/torch_library/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture")
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_LIBRARY_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR})

find_package(ASC REQUIRED)
project(kernel_pytorch_call_torch_library LANGUAGES ASC CXX)
find_package(Python3 COMPONENTS Interpreter Development REQUIRED)

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import torch; print(torch.utils.cmake_prefix_path)"
    OUTPUT_VARIABLE TORCH_CMAKE_PREFIX_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
    RESULT_VARIABLE TORCH_QUERY_RESULT
)
if(NOT TORCH_QUERY_RESULT EQUAL 0)
    message(FATAL_ERROR "Failed to query Torch CMake prefix")
endif()
string(REPLACE "\r" "\n" TORCH_CMAKE_PREFIX_PATH_LINES "${TORCH_CMAKE_PREFIX_PATH_RAW}")
string(REGEX MATCH "[^\n]+$" TORCH_CMAKE_PREFIX_PATH "${TORCH_CMAKE_PREFIX_PATH_LINES}")
find_package(Torch REQUIRED HINTS ${TORCH_CMAKE_PREFIX_PATH})

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import os, torch_npu; print(os.path.dirname(torch_npu.__file__))"
    OUTPUT_VARIABLE TORCH_NPU_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
    RESULT_VARIABLE TORCH_NPU_QUERY_RESULT
)
if(NOT TORCH_NPU_QUERY_RESULT EQUAL 0)
    message(FATAL_ERROR "Failed to query torch_npu path")
endif()
string(REPLACE "\r" "\n" TORCH_NPU_PATH_LINES "${TORCH_NPU_PATH_RAW}")
string(REGEX MATCH "[^\n]+$" TORCH_NPU_PATH "${TORCH_NPU_PATH_LINES}")
set(TORCH_NPU_INCLUDE_DIRS ${TORCH_NPU_PATH}/include)
set(TORCH_NPU_LIBRARIES ${TORCH_NPU_PATH}/lib)

add_library(ascendc_kernels SHARED
    ${CMAKE_CURRENT_LIST_DIR}/../common/add_custom.asc
)

add_library(ascendc_ops SHARED
    torch_binding.cpp
)

target_include_directories(ascendc_ops SYSTEM PRIVATE
    ${TORCH_INCLUDE_DIRS}
    ${TORCH_NPU_INCLUDE_DIRS}
)

target_link_directories(ascendc_ops PRIVATE
    ${TORCH_NPU_LIBRARIES}
)

target_link_libraries(ascendc_ops PRIVATE
    ascendc_kernels
    torch_npu
    Python3::Python
)

target_compile_options(ascendc_kernels PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

target_compile_options(ascendc_ops PRIVATE
    ${TORCH_CXX_FLAGS}
)

set_target_properties(ascendc_ops PROPERTIES
    BUILD_RPATH "$ORIGIN"
    INSTALL_RPATH "$ORIGIN"
)


下面直接完成 torch.library 样例的配置、编译和运行。同样直接调用 `cmake` 与 `make`，运行测试前把 `build` 目录加入 `LD_LIBRARY_PATH`。


#### 3.1.5 python侧调用代码

torch.library 方式在 Python 侧的关键动作是：

1. 使用 `torch.ops.load_library("libascendc_ops.so")` 加载动态库；
2. 通过 `torch.ops.ascendc_ops.ascendc_add(...)` 调用自定义算子；
3. 将结果与 CPU 标准结果比较。

本节会进入 `build` 目录执行测试，并将 `build` 目录加入动态库搜索路径，因此 Python 侧可以和参考样例一样直接按 so 名称加载。与 pybind11 方式相比，这里没有 `import ascendc_ops`，因为算子是注册到 PyTorch dispatcher 中的。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/torch_library/add_custom_test.py
import torch
import torch_npu
from torch_npu.testing.testcase import TestCase, run_tests

torch.ops.load_library("libascendc_ops.so")


class TestTorchLibraryCustomAdd(TestCase):
    def test_add_custom_ops(self):
        torch.manual_seed(0)
        shape = (8, 2048)
        x = torch.rand(shape, device="cpu", dtype=torch.float16)
        y = torch.rand(shape, device="cpu", dtype=torch.float16)

        output = torch.ops.ascendc_ops.ascendc_add(x.npu(), y.npu()).cpu()
        expected = torch.add(x, y)
        self.assertRtolEqual(output, expected)


if __name__ == "__main__":
    run_tests()


In [ ]:
!cd src/02.06/kernel_pytorch_call/torch_library && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 -DPython3_EXECUTABLE="$PY_BIN" .. && \
make -j && \
LD_LIBRARY_PATH="$PWD:${LD_LIBRARY_PATH:-}" "$PY_BIN" ../add_custom_test.py

## 4. 课程小节

本节完成了两条 PyTorch Kernel 直调路径：

| 调用方式 | C++侧入口 | Python侧调用 | 适用特点 |
| --- | --- | --- | --- |
| pybind11 | `PYBIND11_MODULE` 绑定普通 Python 模块函数 | `import ascendc_ops` 后调用 `ascendc_ops.ascendc_add(...)` | 接入简单，适合快速验证 C++ 封装函数 |
| torch.library | `TORCH_LIBRARY` 定义 schema，`TORCH_LIBRARY_IMPL` 注册 NPU 实现 | `torch.ops.load_library(...)` 后调用 `torch.ops.ascendc_ops.ascendc_add(...)` | 接入 PyTorch 算子系统，适合作为框架算子适配方式 |

两种方式的共同点是：输入输出都使用 PyTorch Tensor，封装层从 Tensor 中取得 device 指针，拿到当前 NPU stream 后通过 `<<<>>>` 发起 Ascend C Kernel 调用。


## 5. 课程实践

请完成一个 torch.library 方式的 Sub 向量减法算子集成题：在 `src/02.06/kernel_pytorch_call/practice_torch_library` 中补全 `practice_sub` 算子。

本实践参考 Ascend C SIMD API 中的 `Sub` 接口。Kernel 侧代码 `sub_custom.asc` 已直接给出，它使用 TPipe/TQue 完成输入搬运、UB 内 `AscendC::Sub` 计算和结果写回。练习重点放在 PyTorch 框架集成的关键位置：

- 在 `torch_binding.cpp` 中补全 Tensor 参数检查、当前 NPU stream 获取、输出 Tensor 创建、`sub_custom_impl(...)` 调用，以及 torch.library 注册；
- 在 `CMakeLists.txt` 中补全 Kernel 库、torch.library 注册库、include/link/RPATH 等关键编译配置；
- 在 `practice_test.py` 中补全动态库加载、`torch.ops.practice_sub_ops.practice_sub(...)` 调用和结果校验。

题目固定支持 `x: [8, 2048] float16` 与 `y: [8, 2048] float16`，输出为 `[8, 2048] float16`。下面先生成可补全的工程骨架，参考答案单独放在本课程目录的 `answer/02.06` 下，notebook 执行时会复制这份答案到运行目录并直接编译运行。


In [ ]:
import shutil
from pathlib import Path

ROOT = Path("src/02.06/kernel_pytorch_call")
PRACTICE_DIR = ROOT / "practice_torch_library"
ANSWER_DIR = ROOT / "answer" / "02.06"
STANDALONE_ANSWER_DIR = Path("answer/02.06")

PRACTICE_DIR.mkdir(parents=True, exist_ok=True)
if ANSWER_DIR.exists():
    shutil.rmtree(ANSWER_DIR)
shutil.copytree(STANDALONE_ANSWER_DIR, ANSWER_DIR, ignore=shutil.ignore_patterns("build"))
shutil.copy2(STANDALONE_ANSWER_DIR / "sub_custom.asc", PRACTICE_DIR / "sub_custom.asc")
print(f"题目目录: {PRACTICE_DIR}")
print(f"独立参考答案目录: {STANDALONE_ANSWER_DIR}")
print(f"本次运行复制目录: {ANSWER_DIR}")


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/practice_torch_library/torch_binding.cpp
#include <cstdint>
#include <torch/extension.h>
#include "torch_npu/csrc/core/npu/NPUStream.h"

extern "C" void sub_custom_impl(uint32_t numBlocks, void* aclStream, uint8_t* x, uint8_t* y, uint8_t* z,
                                uint32_t totalLength);

namespace {
constexpr int64_t SUB_ROWS = 8;
constexpr int64_t SUB_COLS = 2048;
}

namespace practice_sub_ops {
at::Tensor practice_sub(const at::Tensor& x, const at::Tensor& y)
{
    TORCH_CHECK(x.dim() == 2 && x.size(0) == SUB_ROWS && x.size(1) == SUB_COLS,
        "practice_sub only supports lhs shape [8, 2048].");
    TORCH_CHECK(y.dim() == 2 && y.size(0) == SUB_ROWS && y.size(1) == SUB_COLS,
        "practice_sub only supports rhs shape [8, 2048].");
    TORCH_CHECK(x.scalar_type() == at::ScalarType::Half && y.scalar_type() == at::ScalarType::Half,
        "practice_sub only supports float16 tensors.");
    TORCH_CHECK(x.device() == y.device(), "lhs and rhs must be on the same device.");
    TORCH_CHECK(x.is_contiguous() && y.is_contiguous(), "practice_sub only supports contiguous tensors.");

    // TODO: 获取当前 NPU stream。
    // TODO: 按输入 shape 和 options 创建输出 Tensor。
    // TODO: 调用 sub_custom_impl(...)，传入核数、stream、x/y/z 的 device 指针和 totalLength。
    // TODO: 返回输出 Tensor。
}
}  // namespace practice_sub_ops

// TODO: 使用 TORCH_LIBRARY 定义 practice_sub_ops::practice_sub 的 schema。

// TODO: 使用 TORCH_LIBRARY_IMPL 将 practice_sub 注册到 PrivateUse1。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/practice_torch_library/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture")
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_LIBRARY_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR})

find_package(ASC REQUIRED)
project(practice_sub_torch_library LANGUAGES ASC CXX)
find_package(Python3 COMPONENTS Interpreter Development REQUIRED)

function(select_probe_line output_var raw_output suffix)
    string(REPLACE "
" ";" output_lines "${raw_output}")
    foreach(output_line IN LISTS output_lines)
        string(STRIP "${output_line}" output_line)
        if(output_line MATCHES "${suffix}$")
            set(${output_var} "${output_line}" PARENT_SCOPE)
            return()
        endif()
    endforeach()
    message(FATAL_ERROR "Cannot find path ending with ${suffix} in Python probe output: ${raw_output}")
endfunction()

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import torch; print(torch.utils.cmake_prefix_path)"
    OUTPUT_VARIABLE TORCH_CMAKE_PREFIX_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
)
select_probe_line(TORCH_CMAKE_PREFIX_PATH "${TORCH_CMAKE_PREFIX_PATH_RAW}" "/torch/share/cmake")
find_package(Torch REQUIRED HINTS ${TORCH_CMAKE_PREFIX_PATH})

execute_process(
    COMMAND ${Python3_EXECUTABLE} -c "import os, torch_npu; print(os.path.dirname(torch_npu.__file__))"
    OUTPUT_VARIABLE TORCH_NPU_PATH_RAW
    OUTPUT_STRIP_TRAILING_WHITESPACE
)
select_probe_line(TORCH_NPU_PATH "${TORCH_NPU_PATH_RAW}" "/torch_npu")
set(TORCH_NPU_INCLUDE_DIRS ${TORCH_NPU_PATH}/include)
set(TORCH_NPU_LIBRARIES ${TORCH_NPU_PATH}/lib)

# TODO: add_library(practice_sub_kernels SHARED sub_custom.asc)
# TODO: add_library(practice_sub_ops SHARED torch_binding.cpp)
# TODO: 为 practice_sub_ops 添加 Torch 与 torch_npu include 目录。
# TODO: 为 practice_sub_ops 添加 torch_npu lib 目录。
# TODO: 链接 practice_sub_kernels、torch_npu、Python3::Python。
# TODO: 为 practice_sub_kernels 添加 --npu-arch=${CMAKE_ASC_ARCHITECTURES} 编译选项。
# TODO: 设置 practice_sub_ops 的 BUILD_RPATH/INSTALL_RPATH 为 $ORIGIN。


In [ ]:
%%writefile src/02.06/kernel_pytorch_call/practice_torch_library/practice_test.py
import torch
import torch_npu
from torch_npu.testing.testcase import TestCase, run_tests

# TODO: 使用 torch.ops.load_library("libpractice_sub_ops.so") 加载动态库。


class TestPracticeSub(TestCase):
    def test_practice_sub(self):
        torch.manual_seed(0)
        x = torch.rand([8, 2048], device="cpu", dtype=torch.float16)
        y = torch.rand([8, 2048], device="cpu", dtype=torch.float16)

        # TODO: 调用 torch.ops.practice_sub_ops.practice_sub(x.npu(), y.npu()).cpu()。
        # TODO: 使用 x - y 生成期望结果。
        # TODO: 使用 self.assertRtolEqual(...) 校验结果。
        raise NotImplementedError("请补全 practice_sub 的 torch.library 集成调用")


if __name__ == "__main__":
    run_tests()


补全上述 `torch_binding.cpp`、`CMakeLists.txt` 和 `practice_test.py` 中的 TODO 后，运行下面的命令直接配置、编译并运行练习工程验证结果。


In [ ]:
!cd src/02.06/kernel_pytorch_call/practice_torch_library && \
rm -rf build && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 -DPython3_EXECUTABLE="$PY_BIN" .. && \
make -j && \
LD_LIBRARY_PATH="$PWD:${LD_LIBRARY_PATH:-}" "$PY_BIN" ../practice_test.py

### 5.1 参考答案

参考答案已作为独立目录放在 `answer/02.06`。下面直接编译运行复制到 `src/02.06/kernel_pytorch_call/answer/02.06` 的参考答案，验证完整 Sub torch.library 集成流程。


In [ ]:
!cd src/02.06/kernel_pytorch_call/answer/02.06 && \
rm -rf build && \
mkdir -p build && \
cd build && \
cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 -DPython3_EXECUTABLE="$PY_BIN" .. && \
make -j && \
LD_LIBRARY_PATH="$PWD:${LD_LIBRARY_PATH:-}" "$PY_BIN" ../practice_test.py